# Strategy Analysis with Backtrader

This notebook demonstrates how to analyze trading strategies using the Backtrader integration.

In [ ]:
# Import required libraries
import backtrader as bt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add the project root to the Python path
sys.path.append(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('')))))

# Import our custom modules
from backtrader.data.freqtrade_converter import FreqTradeConverter
from backtrader.data.data_loader import BacktraderDataLoader
from backtrader.strategies.scalping_strategy import ScalpingStrategy
from backtrader.strategies.scalping_july_strategy import ScalpingJulyStrategy

## Load and Convert Data

In [ ]:
# Initialize converters
converter = FreqTradeConverter()
loader = BacktraderDataLoader()

# Load BTC/USDT 1m data
data_file = "../../user_data/data/binance/BTC_USDT-1m.json"

if os.path.exists(data_file):
    print(f"Loading data from {data_file}")
    data_feed = loader.load_from_freqtrade_json(data_file, "BTC_USDT")
else:
    print(f"Data file {data_file} not found. Creating sample data.")
    # Create sample data for demonstration
    dates = pd.date_range('2023-01-01', periods=1000, freq='1min')
    prices = 100 + np.cumsum(np.random.randn(1000) * 0.1)
    df = pd.DataFrame({
        'open': prices,
        'high': prices + np.random.rand(1000) * 2,
        'low': prices - np.random.rand(1000) * 2,
        'close': prices + np.random.randn(1000) * 0.5,
        'volume': np.random.rand(1000) * 1000
    }, index=dates)
    
    data_feed = loader.load_from_dataframe(df, "SampleData")

print(f"Data loaded with {len(data_feed)} bars")

## Run Backtest

In [ ]:
# Create a cerebro entity
cerebro = bt.Cerebro()

# Set up initial cash
cerebro.broker.setcash(1000.0)

# Set commission (0.1% per trade)
cerebro.broker.setcommission(commission=0.001)

# Set position size
cerebro.broker.setpositionSizer(bt.sizers.PercentSizer, percents=10)  # 10% of portfolio per trade

# Add data
cerebro.adddata(data_feed)

# Add strategy
cerebro.addstrategy(ScalpingStrategy)

# Print out the starting conditions
print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())

# Run over everything
results = cerebro.run()

# Print out the final result
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())

## Analyze Results

In [ ]:
# Calculate performance metrics
start_value = 1000.0
end_value = cerebro.broker.getvalue()
total_return = (end_value - start_value) / start_value * 100

print(f"Total Return: {total_return:.2f}%")
print(f"Profit: ${end_value - start_value:.2f}")

## Visualize Results

In [ ]:
# Plot the result
cerebro.plot()

## Compare Strategies

In [ ]:
# Strategies to compare
strategies = [
    (ScalpingStrategy, "ScalpingStrategy"),
    (ScalpingJulyStrategy, "ScalpingJulyStrategy")
]

# Results storage
results = []

for strategy_class, strategy_name in strategies:
    # Create a cerebro entity
    cerebro = bt.Cerebro()
    
    # Set up initial cash
    cerebro.broker.setcash(1000.0)
    
    # Set commission (0.1% per trade)
    cerebro.broker.setcommission(commission=0.001)
    
    # Set position size
    cerebro.broker.setpositionSizer(bt.sizers.PercentSizer, percents=10)
    
    # Add data
    cerebro.adddata(data_feed)
    
    # Add strategy
    cerebro.addstrategy(strategy_class)
    
    # Run backtest
    strat_results = cerebro.run()
    
    # Store results
    results.append({
        'strategy': strategy_name,
        'start_value': 1000.0,
        'end_value': cerebro.broker.getvalue(),
        'return_pct': (cerebro.broker.getvalue() - 1000.0) / 1000.0 * 100
    })

# Display comparison
print("Strategy Comparison:")
print("-" * 40)
for result in results:
    print(f"{result['strategy']}: {result['return_pct']:.2f}%")